# E-Commerce Logistics & Sales Analyzer
#### The Scenario: You are given three separate datasets: "Customer Profiles" (age, region, subscription tier), "Product Catalog" (category, weight, unit price), and "Order Transactions" (who bought what, when, and delivery times).

In [2]:
import pandas as pd
import numpy as np

jan_orders = pd.read_csv('orders_jan.csv')
feb_orders = pd.read_csv('orders_feb.csv')
products = pd.read_csv('products.csv')
customers = pd.read_csv('customers.csv')

print("Datasets Successfully Load")

Datasets Successfully Load


## Combining Datasets
#### You must use pd.concat() to stitch together the order transactions from "January" and "February". Then, use pd.merge() to join the combined 'Order Transactions' table with the 'Product Catalog' table so you know the category and price of the item purchased, acting like a SQL left join.

In [12]:
orders = pd.concat([jan_orders, feb_orders], ignore_index=True)

orders_merged = pd.merge(orders, products, on='Product_ID', how='left')

df = pd.merge(orders_merged, customers, left_on='cust_ID', right_on='Customer_ID', how='left')

df

,Transaction_ID,cust_ID,Product_ID,Quantity,Order_Date,Delivery_Days,Warehouse_Route_ID,Product_Category,Weight_kg,Unit_Price,Customer_ID,Age,Region,Subscription_Tier
0,T00001,C0546,P0146,4,2024-01-26,2,WH-A1,Home & Garden,11.97,403.69,C0546,70,South,Premium
1,T00002,C0609,P0938,1,2024-01-22,6,WH-C4,Toys,11.11,152.34,C0609,41,North,Basic
2,T00003,C0169,P0057,3,2024-01-07,6,WH-B3,Clothing,10.97,8.16,C0169,49,East,Premium
3,T00004,C0681,P0723,1,2024-01-10,1,WH-A9,Electronics,11.15,488.40,C0681,42,South,Basic
4,T00005,C0663,P0780,3,2024-01-28,1,WH-A7,Sports,7.34,476.43,C0663,52,South,Basic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,T01996,C0637,P0623,3,2024-02-19,3,WH-C2,Toys,7.38,298.92,C0637,43,East,Plus
1996,T01997,C0716,P0210,3,2024-02-09,4,WH-A8,Toys,4.08,55.91,C0716,70,North,Basic
1997,T01998,C0573,P0513,1,2024-02-09,1,WH-B4,Toys,2.94,85.87,C0573,33,North,Basic
1998,T01999,C0762,P0058,3,2024-02-13,2,WH-C9,Home & Garden,14.51,489.29,C0762,24,North,Premium


## Modifying DataFrames
#### You create a new calculated column called Total_Revenue by multiplying the Quantity column by the Unit_Price column. Use .drop() to remove redundant warehouse routing IDs, and use .rename() to fix inconsistently capitalized columns (e.g., changing cust_ID to Customer_ID).

In [13]:
df['Total_Revenue'] = df['Quantity'] * df['Unit_Price']

df = df.drop(columns=['Warehouse_Route_ID', 'Customer_ID'])

df = df.rename(columns={'cust_ID': 'Customer_ID'})
df.head()

,Transaction_ID,Customer_ID,Product_ID,Quantity,Order_Date,Delivery_Days,Product_Category,Weight_kg,Unit_Price,Age,Region,Subscription_Tier,Total_Revenue
0,T00001,C0546,P0146,4,2024-01-26,2,Home & Garden,11.97,403.69,70,South,Premium,1614.76
1,T00002,C0609,P0938,1,2024-01-22,6,Toys,11.11,152.34,41,North,Basic,152.34
2,T00003,C0169,P0057,3,2024-01-07,6,Clothing,10.97,8.16,49,East,Premium,24.48
3,T00004,C0681,P0723,1,2024-01-10,1,Electronics,11.15,488.40,42,South,Basic,488.40
4,T00005,C0663,P0780,3,2024-01-28,1,Sports,7.34,476.43,52,South,Basic,1429.29


## Applying Functions
#### You write a custom function (or use a lambda) and apply it via .apply() to categorize the Delivery_Days column into delivery performance buckets: "Early" (1-2 days), "On-Time" (3-4 days), or "Delayed" (5+ days).

In [15]:
def categorize_delivery(days):
    if days <= 2:
        return 'Early'
    elif days <= 4:
        return 'On-Time'
    else:
        return 'Delayed'

df['Delivery Performance'] = df['Delivery_Days'].apply(categorize_delivery)
df.head()

,Transaction_ID,Customer_ID,Product_ID,Quantity,Order_Date,Delivery_Days,Product_Category,Weight_kg,Unit_Price,Age,Region,Subscription_Tier,Total_Revenue,Delivery Performance
0,T00001,C0546,P0146,4,2024-01-26,2,Home & Garden,11.97,403.69,70,South,Premium,1614.76,Early
1,T00002,C0609,P0938,1,2024-01-22,6,Toys,11.11,152.34,41,North,Basic,152.34,Delayed
2,T00003,C0169,P0057,3,2024-01-07,6,Clothing,10.97,8.16,49,East,Premium,24.48,Delayed
3,T00004,C0681,P0723,1,2024-01-10,1,Electronics,11.15,488.40,42,South,Basic,488.40,Early
4,T00005,C0663,P0780,3,2024-01-28,1,Sports,7.34,476.43,52,South,Basic,1429.29,Early


## Grouping and Aggregation
####  Using .groupby(), you group the data by Product_Category and use .agg() to find the total revenue generated, the average delivery days, and the unique count of Customer_IDs who purchased from that category.

In [21]:
category_stats = df.groupby('Product_Category').agg(
    Total_Revenue=('Total_Revenue', 'sum'),
    Avg_Delivery_Days=('Delivery_Days', 'mean'),
    Unique_Customers=('Customer_ID', 'nunique')
).reset_index()
category_stats

,Product_Category,Total_Revenue,Avg_Delivery_Days,Unique_Customers
0,Clothing,236216.30,4.005764,289
1,Electronics,291529.29,3.960688,333
2,Home & Garden,318433.12,4.009324,342
3,Sports,311298.28,3.847255,346
4,Toys,300972.12,3.871859,316


## Reshaping and Pivoting
#### Finally, you use pd.pivot_table() to create a business-reporting matrix showing "Customer Region" as rows, "Delivery Performance" (Early/On-Time/Delayed) as columns, and the "Total Revenue" as the values.

In [30]:
pivot_matrix = pd.pivot_table(
    df, 
    values='Total_Revenue', 
    index='Region', 
    columns='Delivery Performance', 
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name = 'Total Revenue'
)
pivot_matrix.style.format("${:,.2f}").background_gradient(cmap='Greens')

Delivery Performance,Delayed,Early,On-Time,Total Revenue
Region,,,,
East,"$165,532.82","$91,495.83","$102,829.30","$359,857.95"
North,"$168,913.17","$115,843.80","$125,378.86","$410,135.83"
South,"$157,531.94","$112,139.58","$102,640.26","$372,311.78"
West,"$129,061.42","$111,503.35","$75,578.78","$316,143.55"
Total Revenue,"$621,039.35","$430,982.56","$406,427.20","$1,458,449.11"
